# File Organizer

In questo notebook eseguiremo tre passi:

1. **Organizzare** tutti i file in sottocartelle per estensione  
2. **Analizzare** ricorsivamente tutte le immagini


In [9]:
import os
import shutil
import csv
from PIL import Image, UnidentifiedImageError
import numpy as np
from tabulate import tabulate
import glob

CARTELLA_BASE = r"./FileOrganizer"
CARTELLA_FILE = os.path.join(CARTELLA_BASE, 'files')
FILE_RIEPILOGO = os.path.join(CARTELLA_BASE, 'recap.csv')
FILE_RIEPILOGO_IMMAGINI = os.path.join(CARTELLA_BASE, 'recap_images.csv')

# Mappatura delle estensioni alle categorie di file
CATEGORIE = {
    'audio': {'mp3', 'wav', 'flac', 'aac', 'ogg', 'm4a'},
    'immagini': {'jpg', 'jpeg', 'png', 'gif', 'bmp', 'tiff', 'svg'},
    'documenti': {'pdf', 'doc', 'docx', 'odt', 'txt', 'csv', 'xls', 'xlsx', 'ppt', 'pptx'},
}


## Passo 1 – organizzazione di tutti i file

In [11]:
def organizza_tutto(cartella_file=CARTELLA_FILE, file_riepilogo=FILE_RIEPILOGO):
    """
    1. Crea recap.csv se non esiste (con intestazione).
    2. Raggruppa i file in `cartella_file` per categoria (audio, immagini, documenti, altro).
    3. Per ogni categoria, ordina alfabeticamente i nomi file e sposta i file nella sottocartella corrispondente.
    4. Registra in CSV nome file, percorso vecchio e percorso nuovo nell'ordine di spostamento.
    """
    intestazione = ['nome_file', 'percorso_vecchio', 'percorso_nuovo']
    # Crea il file di riepilogo se non esiste
    if not os.path.exists(file_riepilogo):
        with open(file_riepilogo, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(intestazione)

    # Raggruppa i file per categoria
    files_per_cat = {cat: [] for cat in CATEGORIE}
    files_per_cat['altro'] = []
    for nome in os.listdir(cartella_file):
        percorso = os.path.join(cartella_file, nome)
        # Salta directory e file di riepilogo
        if not os.path.isfile(percorso) or os.path.abspath(percorso) == os.path.abspath(file_riepilogo):
            continue
        est = os.path.splitext(nome)[1].lstrip('.').lower()
        # Determina categoria
        if est in CATEGORIE['audio']:
            cat = 'audio'
        elif est in CATEGORIE['immagini']:
            cat = 'immagini'
        elif est in CATEGORIE['documenti']:
            cat = 'documenti'
        else:
            cat = 'altro'
        files_per_cat[cat].append(nome)

    # Sposta i file per ogni categoria in ordine alfabetico
    for cat, nomi in files_per_cat.items():
        if not nomi:
            continue
        dest_dir = os.path.join(cartella_file, cat)
        os.makedirs(dest_dir, exist_ok=True)
        for nome in sorted(nomi, key=lambda s: s.lower()):
            vecchio = os.path.join(cartella_file, nome)
            nuovo = os.path.join(dest_dir, nome)
            shutil.move(vecchio, nuovo)
            # Registra nel CSV
            with open(file_riepilogo, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([nome, vecchio, nuovo])

    print("Passo 1 completato: organizzazione per categorie con spostamento ordinato alfabeticamente.")



In [ ]:
"""
step 2

secondo le specifiche, questo step dovrebbe contenere tutte le logiche dello step 1 (capire la
cartella destinazione, creare o aggiornare il recap...); l'unica differenza è che lo script
sposta un unico file inserito a mano dall'utente via CLI

---

una dritta per semplificare il codice: 

attraverso l'argomento "choices" di "add_argument"
(valorizzato dinamicamente, per esempio con os.listdir) puoi informare 
facilmente l'utente su tutti e soli i file che può effettivamente spostare
(ed evitare il controllo manuale sull'argomento passato da linea di comando)

quindi, per esempio: 

parser.add_argument('pippo', choices=['a', 'b', 'c']) 

così 'pippo' è obbligatorio e può valere solo 'a', 'b' o 'c'

se poi choices è valorizzato con os.listdir, volendo sorted, la lista è dinamica 
"""

## Passo 2 – analisi ricorsiva delle immagini

In [16]:
def analizza_immagini(cartella_file=CARTELLA_FILE, file_riepilogo=FILE_RIEPILOGO_IMMAGINI):
    """
    - Analizza tutti i file immagine riconosciuti da PIL nelle sottocartelle di `cartella_file`.
    - Per ciascuna immagine calcola:
        • ndim (numero di dimensioni dell'array)
        • shape (dimensioni dell'array)
        • media dei pixel per ogni canale (scala di grigio, R, G, B, A)
    - Stampa la tabella in console (usando tabulate con floatfmt) e salva recap_images.csv.
    """
    # Ottiene tutti i percorsi dei file nelle sottocartelle (un solo ciclo)
    pattern = os.path.join(cartella_file, '*', '*')
    percorsi = glob.glob(pattern)

    risultati = []
    max_canali = 1

    for percorso in percorsi:
        if not os.path.isfile(percorso):
            continue
        try:
            img = Image.open(percorso)
            arr = np.array(img)
        except (UnidentifiedImageError, IOError):
            continue

        ndim = arr.ndim
        shape = arr.shape

        # Calcola media pixel per canale
        if ndim == 2:
            # Scala di grigio
            mean_vals = [arr.mean()]
        elif ndim == 3:
            # RGB o RGBA
            mean_vals = arr.mean(axis=(0, 1)).tolist()
        else:
            # Altri casi (es. immagini volumetriche)
            mean_vals = [arr.mean()]

        max_canali = max(max_canali, len(mean_vals))
        risultati.append({
            'immagine': os.path.relpath(percorso, cartella_file),
            'ndim': ndim,
            'shape': shape,
            'mean_vals': mean_vals
        })

    if risultati:
        # Costruisce la tabella con colonne per ogni canale
        tabella = []
        for item in risultati:
            riga = {
                'immagine': item['immagine'],
                'ndim': item['ndim'],
                'shape': item['shape']
            }
            for i in range(max_canali):
                key = f'media_canale_{i}'
                riga[key] = item['mean_vals'][i] if i < len(item['mean_vals']) else None
            tabella.append(riga)

        # Stampa in console con formattazione dei float
        print(tabulate(tabella, headers="keys", tablefmt="github", floatfmt=".2f"))

        # Salva il recap in CSV
        fieldnames = ['immagine', 'ndim', 'shape'] + [f'media_canale_{i}' for i in range(max_canali)]
        with open(file_riepilogo, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(tabella)

        print("Passo 3 completato: recap immagini salvato.")
    else:
        print("Nessuna immagine trovata.")


## Richiamo funzione organizza_tutto

In [20]:
organizza_tutto()

Passo 1 completato: organizzazione per categorie con spostamento ordinato alfabeticamente.


## Richiamo funzione analizza_immagini

In [22]:
analizza_immagini()

| immagine              |   ndim | shape         |   media_canale_0 |   media_canale_1 |   media_canale_2 |   media_canale_3 |
|-----------------------|--------|---------------|------------------|------------------|------------------|------------------|
| immagini\bw.png       |      2 | (512, 512)    |            21.48 |                  |                  |                  |
| immagini\daffodil.jpg |      3 | (500, 335, 3) |           109.25 |            85.56 |             4.97 |                  |
| immagini\eclipse.png  |      3 | (256, 256, 4) |           109.05 |           109.52 |            39.85 |           133.59 |
| immagini\trump.jpeg   |      3 | (183, 275, 3) |            97.01 |            98.99 |            90.92 |                  |
Passo 3 completato: recap immagini salvato.
